# COMPOSE — Final Notebook
**Compositional Object Manipulation via Semantic Embeddings**

### Before running
- Runtime → Change runtime type → **T4 GPU** → Save
- Run **Cell 1 first** — it installs everything and auto-restarts the kernel
- After the kernel restarts, run **Cells 2 through 6** in order
- Do NOT re-run Cell 1 after the restart

In [1]:
# ── CELL 1: Install + auto restart kernel ────────────────────────────────────
# Run this cell ONCE. It installs everything then restarts the kernel.
# After restart, run Cells 2-6. Do NOT re-run this cell.
import os

os.system('pip install numpy==1.26.4 -q --force-reinstall')
os.system('pip install "gradio>=3.50,<4.0" transformers huggingface_hub torch torchvision ultralytics pybullet matplotlib Pillow scipy -q')

print('All packages installed. Restarting kernel now...')
print('After restart: run Cells 2 through 6 in order.')

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

All packages installed. Restarting kernel now...
After restart: run Cells 2 through 6 in order.


{'status': 'ok', 'restart': True}

In [1]:
# ── CELL 2: Clone repo and set path ──────────────────────────────────────────
# Verify numpy version first
import numpy as np
print('numpy version:', np.__version__)  # must be 1.26.4

import os, sys
REPO = 'compose-axion-ax-hackathon-2026-full-solution-template'

if not os.path.exists(f'/content/{REPO}'):
    os.system(f'git clone https://github.com/k2406/{REPO}.git')
else:
    os.system(f'cd /content/{REPO} && git pull origin main')

SRC = f'/content/{REPO}/src'
os.chdir(SRC)
sys.path.insert(0, SRC)
print('Working dir:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

numpy version: 1.26.4
Working dir: /content/compose-axion-ax-hackathon-2026-full-solution-template/src
Files: ['app.py', 'core', 'evaluate.py', 'gui', 'perception.py', 'requirements.txt', 'results.csv', 'simulation', 'train_attribute_mlp.py']


In [3]:
# ── CELL 3: Fix perception.py + Train MLP ────────────────────────────────────
import subprocess

# Fix Shape.BLOCK reference in perception.py
!sed -i 's/Shape\.BLOCK/Shape.CUBE/g' /content/compose-axion-ax-hackathon-2026-full-solution-template/src/perception.py
!sed -i 's/Shape\.BLOCK/Shape.CUBE/g' /content/compose-axion-ax-hackathon-2026-full-solution-template/src/core/reasoning.py
print("Shape.BLOCK references fixed")

# Train MLP
from perception import train_attribute_mlp
print('Training attribute MLP on GPU...')
mlp = train_attribute_mlp(save_path='mlp_weights.pth', n_samples=800, epochs=40)
print('MLP trained and saved to mlp_weights.pth')

Shape.BLOCK references fixed
Training attribute MLP on GPU...
[MLP] Generating 800 synthetic samples...
  Epoch 10/40  loss=1.7977
  Epoch 20/40  loss=0.9016
  Epoch 30/40  loss=0.5638
  Epoch 40/40  loss=0.3931
[MLP] Weights saved to mlp_weights.pth
MLP trained and saved to mlp_weights.pth


In [5]:
!sed -i 's/Shape\.BLOCK/Shape.CUBE/g' /content/compose-axion-ax-hackathon-2026-full-solution-template/src/simulation/pybullet_env.py
print("Fixed pybullet_env.py")

from simulation.pybullet_env import BulletEnv
from core.scene import make_default_scene
import numpy as np

bullet = BulletEnv()
bullet.start()
scene0 = make_default_scene()
bullet.load_scene(scene0)

frame = bullet.capture_frame()
if frame is not None:
    print(f'PyBullet rendering: OK — frame shape: {frame.shape}')
else:
    print('PyBullet: falling back to matplotlib renderer')

Fixed pybullet_env.py
PyBullet rendering: OK — frame shape: (420, 640, 3)


In [6]:
import os
os.chdir('/content/compose-axion-ax-hackathon-2026-full-solution-template')
!git add src/perception.py src/simulation/pybullet_env.py src/core/reasoning.py
!git commit -m "fix: remove all Shape.BLOCK references, replace with Shape.CUBE"
!git push origin main

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@6216282554b8.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


In [10]:
# ── CELL 4: Load YOLO + DINOv2 + PyBullet ────────────────────────────────────
import numpy as np

from perception import Perceptor
perceptor = Perceptor(mlp_weights='mlp_weights.pth')
perceptor.load()
print('Perceptor ready — YOLO + DINOv2 + MLP loaded on CUDA')

from simulation.pybullet_env import BulletEnv
from core.scene import make_default_scene

bullet = BulletEnv()
bullet.start()
scene0 = make_default_scene()
bullet.load_scene(scene0)

frame = bullet.capture_frame()
if frame is not None:
    print(f'PyBullet rendering: OK — frame shape: {frame.shape}')
else:
    print('PyBullet: falling back to matplotlib renderer')

[Perceptor] Loading on cuda
[Perceptor] Loading YOLOv8m...
[Perceptor] Loading DINOv2-large (frozen)...


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

[Perceptor] Loading MLP weights from mlp_weights.pth
[Perceptor] Ready
Perceptor ready — YOLO + DINOv2 + MLP loaded on CUDA
PyBullet rendering: OK — frame shape: (420, 640, 3)


In [8]:
import os
os.path.exists('/content/compose-axion-ax-hackathon-2026-full-solution-template/src/mlp_weights.pth')

True

In [9]:
from perception import train_attribute_mlp
mlp = train_attribute_mlp(save_path='mlp_weights.pth', n_samples=800, epochs=40)
perceptor.mlp.load_state_dict(__import__('torch').load('mlp_weights.pth'))
print("MLP weights loaded")

[MLP] Generating 800 synthetic samples...
  Epoch 10/40  loss=1.7901
  Epoch 20/40  loss=0.9204
  Epoch 30/40  loss=0.5774
  Epoch 40/40  loss=0.4056
[MLP] Weights saved to mlp_weights.pth
MLP weights loaded


In [11]:
# ── CELL 5: Run benchmark ─────────────────────────────────────────────────────
# Expected: 6/6 PASS, TSR 100%, Novel colour gen 100%
!cd /content/compose-axion-ax-hackathon-2026-full-solution-template/src && python evaluate.py

COMPOSE — Benchmark Evaluation

[PASS] Demo 1 — Baseline
  Command : move red cube right of blue cube
  Result  : Moving red cube right of blue cube → (431, 200)
  Conf    : 0.82   Time: 0.4ms

[PASS] Demo 2 — Attribute reasoning
  Command : move green cylinder behind yellow container
  Result  : Moving green cylinder behind yellow container → (240, 205)
  Conf    : 0.82   Time: 0.2ms

[PASS] Demo 3 — Novel colour generalisation [NOVEL]
  Command : move cyan sphere left of green cylinder
  Result  : Moving cyan sphere left of green cylinder → (388, 140) [NOVEL color — compositio
  Conf    : 0.85   Time: 0.2ms

[PASS] Demo 4 — Novel composition
  Command : move purple container beside red cube
  Result  : Moving purple container beside red cube → (244, 160)
  Conf    : 0.82   Time: 0.2ms

[PASS] Demo 5 — Multi-constraint
  Command : move blue cube in front of yellow container
  Result  : Moving blue cube in front yellow container → (240, 381)
  Conf    : 0.82   Time: 0.2ms

[PASS] Demo 

In [13]:
# ── CELL 6: Launch COMPOSE GUI ───────────────────────────────────────────────
# Expected: public URL printed at bottom e.g. https://xxx.gradio.live

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import gradio as gr

from core.scene import Scene, SceneObject, make_default_scene, Color, Shape, Size
from core.reasoning import reason, NOVEL_COLORS
from gui.renderer import draw_scene, fig_to_pil

print('Gradio version:', gr.__version__)


# ── Metrics tracker ───────────────────────────────────────────────────────────
class Metrics:
    def __init__(self):
        self.total = self.successes = 0
        self.novel_total = self.novel_ok = 0
        self.comp_total  = self.comp_ok  = 0

    def update(self, result):
        self.total += 1
        if result.success: self.successes += 1
        if result.is_novel:
            self.novel_total += 1
            if result.success: self.novel_ok += 1
        if result.intent and result.intent.spatial_rel:
            self.comp_total += 1
            if result.success: self.comp_ok += 1

    @property
    def tsr(self):       return f'{self.successes/self.total*100:.0f}%' if self.total else '—'
    @property
    def novel_gen(self): return f'+{self.novel_ok/self.novel_total*100:.0f}%' if self.novel_total else '—'
    @property
    def comp_gen(self):  return f'{self.comp_ok/self.comp_total*100:.0f}%' if self.comp_total else '—'
    @property
    def goal_acc(self):  return f'{self.comp_ok/self.comp_total*100:.0f}%' if self.comp_total else '—'


# ── State serialisation ───────────────────────────────────────────────────────
def scene_to_dict(scene):
    return {
        'canvas_w': scene.canvas_w, 'canvas_h': scene.canvas_h,
        'objects': [{'obj_id': o.obj_id, 'shape': o.shape.value,
                     'color': o.color.value, 'size': o.size.value,
                     'x': o.x, 'y': o.y} for o in scene.objects],
    }

def dict_to_scene(d):
    return Scene(
        objects=[SceneObject(
            obj_id=o['obj_id'], shape=Shape(o['shape']),
            color=Color(o['color']), size=Size(o['size']),
            x=o['x'], y=o['y'],
        ) for o in d.get('objects', [])],
        canvas_w=d.get('canvas_w', 640),
        canvas_h=d.get('canvas_h', 420),
    )

def metrics_to_dict(m): return vars(m).copy()
def dict_to_metrics(d):
    m = Metrics()
    for k, v in d.items(): setattr(m, k, v)
    return m


# ── Arena rendering — never raises ───────────────────────────────────────────
def get_arena(scene, result=None):
    try:
        frame = bullet.capture_frame()
        if frame is not None:
            return frame
    except Exception:
        pass
    try:
        img = np.array(fig_to_pil(draw_scene(scene, result)))
        plt.close('all')
        return img
    except Exception:
        plt.close('all')
        return np.zeros((420, 640, 3), dtype=np.uint8)


# ── Command handler ───────────────────────────────────────────────────────────
def handle_command(command, history, scene_state, metrics_state):
    if not command.strip():
        return (history, get_arena(dict_to_scene(scene_state)),
                scene_state, metrics_state, '—', '—', '—', '—')

    scene   = dict_to_scene(scene_state)
    metrics = dict_to_metrics(metrics_state)
    result  = reason(command, scene)
    metrics.update(result)

    parts = []
    if result.intent:
        i = result.intent
        chips = ' '.join(filter(None, [
            f'[action:{i.action}]'             if i.action       else None,
            f'[target:{i.target_color.value}]' if i.target_color else None,
            f'[shape:{i.target_shape.value}]'  if i.target_shape else None,
            f'[size:{i.target_size.value}]'    if i.target_size  else None,
            f'[spatial:{i.spatial_rel}]'        if i.spatial_rel  else None,
            f'[conf:{result.confidence}]',
        ]))
        parts.append(f'Parsed: {chips}')

    if result.is_novel:
        parts.append('NOVEL — unseen colour detected. Compositional generalisation applied via disentangled embeddings.')

    if result.ambiguous:
        parts.append(f'Ambiguous — {result.message}')
        arena_img = get_arena(scene)

    elif result.success:
        parts.append(f'Executing: {result.message}')
        dest_x = result.dest_x
        dest_y = result.dest_y
        if dest_x is not None and dest_y is not None:
            obj = scene.get(result.target.obj_id)
            if obj:
                obj.x = dest_x
                obj.y = dest_y
            try:
                ret = bullet.move_object(result.target.obj_id, dest_x, dest_y)
                if isinstance(ret, tuple) and len(ret) == 2:
                    _, pb_ok = ret
                    if pb_ok:
                        parts.append('PyBullet: reached goal position.')
            except Exception as e:
                print(f'[PyBullet fallback] {e}')
        arena_img = get_arena(scene)

    else:
        parts.append(f'Error: {result.message}')
        arena_img = get_arena(scene)

    parts.append(f'TSR: {metrics.tsr}')
    history = history + [[command, '\n'.join(parts)]]
    plt.close('all')

    return (
        history, arena_img,
        scene_to_dict(scene), metrics_to_dict(metrics),
        metrics.tsr, metrics.goal_acc, metrics.novel_gen, metrics.comp_gen,
    )


# ── Add object handler ────────────────────────────────────────────────────────
def handle_add_object(color_str, shape_str, size_str, scene_state, metrics_state):
    scene   = dict_to_scene(scene_state)
    metrics = dict_to_metrics(metrics_state)
    try:
        color = Color(color_str.lower())
        shape = Shape(shape_str.lower())
        size  = Size(size_str.lower())
    except ValueError as e:
        return (scene_to_dict(scene), metrics_to_dict(metrics),
                get_arena(scene), metrics.tsr, metrics.goal_acc,
                metrics.novel_gen, metrics.comp_gen, f'Error: {e}')

    obj_id  = scene.next_obj_id()
    x, y    = scene.free_position()
    new_obj = SceneObject(obj_id=obj_id, shape=shape,
                          color=color, size=size, x=x, y=y)
    scene.objects.append(new_obj)
    try:
        bullet._spawn_object(new_obj)
        bullet._settle(40)
    except Exception:
        pass

    novel_note = ' (NOVEL colour)' if new_obj.is_novel() else ''
    status = f'Added: {new_obj.label} ({size_str}){novel_note} at ({int(x)}, {int(y)})'
    plt.close('all')
    return (
        scene_to_dict(scene), metrics_to_dict(metrics),
        get_arena(scene), metrics.tsr, metrics.goal_acc,
        metrics.novel_gen, metrics.comp_gen, status,
    )


# ── Reset ─────────────────────────────────────────────────────────────────────
def reset_scene(metrics_state):
    scene   = make_default_scene()
    metrics = Metrics()
    try:
        bullet.load_scene(scene)
    except Exception:
        pass
    plt.close('all')
    return (
        [], get_arena(scene),
        scene_to_dict(scene), metrics_to_dict(metrics),
        '—', '—', '—', '—', '',
    )


# ── Demo loader ───────────────────────────────────────────────────────────────
DEMO_COMMANDS = [
    'move red cube right of blue cube',
    'move green cylinder behind yellow container',
    'move cyan sphere left of green cylinder',
    'move purple container beside red cube',
    'move blue cube in front of yellow container',
]

def load_demo(demo_idx, history, scene_state, metrics_state):
    result = handle_command(
        DEMO_COMMANDS[int(demo_idx)], history, scene_state, metrics_state
    )
    return result + ('',)


# ── Initial state ─────────────────────────────────────────────────────────────
scene0   = make_default_scene()
metrics0 = Metrics()
try:
    bullet.load_scene(scene0)
except Exception:
    pass
img0 = get_arena(scene0)

COLORS_LIST = [c.value for c in Color]
SHAPES_LIST = [s.value for s in Shape]
SIZES_LIST  = [s.value for s in Size]


# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(title='COMPOSE') as demo:
    scene_state   = gr.State(scene_to_dict(scene0))
    metrics_state = gr.State(metrics_to_dict(metrics0))

    gr.Markdown("""
# COMPOSE
**Compositional Object Manipulation via Semantic Embeddings**
Type a natural language command to manipulate objects, or add new objects to the scene.
""")

    with gr.Row():
        with gr.Column(scale=3):
            arena = gr.Image(value=img0, show_label=False,
                             height=420, interactive=False)
            with gr.Row():
                tsr_box   = gr.Textbox(value='—', label='Task success',     interactive=False)
                goal_box  = gr.Textbox(value='—', label='Goal accuracy',    interactive=False)
                novel_box = gr.Textbox(value='—', label='Novel colour gen', interactive=False)
                comp_box  = gr.Textbox(value='—', label='Comp. gen',        interactive=False)

            with gr.Accordion('➕  Add object to scene', open=False):
                with gr.Row():
                    color_dd = gr.Dropdown(choices=COLORS_LIST, value='orange', label='Colour')
                    shape_dd = gr.Dropdown(choices=SHAPES_LIST, value='cube',   label='Shape')
                    size_dd  = gr.Dropdown(choices=SIZES_LIST,  value='medium', label='Size')
                add_btn    = gr.Button('Add to scene', variant='primary')
                add_status = gr.Textbox(value='', label='Status', interactive=False)

            with gr.Row():
                demo_btns = [gr.Button(f'Demo {i+1}') for i in range(5)]

            gr.Markdown("""
**Demo commands:**
`1` move red cube right of blue cube
`2` move green cylinder behind yellow container
`3` move cyan sphere left of green cylinder ← **NOVEL colour**
`4` move purple container beside red cube
`5` move blue cube in front of yellow container
""")

        with gr.Column(scale=2):
            chatbot   = gr.Chatbot(height=480)
            cmd_input = gr.Textbox(
                placeholder='move red cube right of blue cube',
                show_label=False)
            send_btn  = gr.Button('Send', variant='primary')
            reset_btn = gr.Button('Reset scene', variant='secondary')

    cmd_outputs   = [chatbot, arena, scene_state, metrics_state,
                     tsr_box, goal_box, novel_box, comp_box]
    add_outputs   = [scene_state, metrics_state, arena,
                     tsr_box, goal_box, novel_box, comp_box, add_status]
    reset_outputs = cmd_outputs + [add_status]

    send_btn.click(
        handle_command,
        inputs=[cmd_input, chatbot, scene_state, metrics_state],
        outputs=cmd_outputs)
    cmd_input.submit(
        handle_command,
        inputs=[cmd_input, chatbot, scene_state, metrics_state],
        outputs=cmd_outputs)
    add_btn.click(
        handle_add_object,
        inputs=[color_dd, shape_dd, size_dd, scene_state, metrics_state],
        outputs=add_outputs)
    reset_btn.click(
        reset_scene,
        inputs=[metrics_state],
        outputs=reset_outputs)
    for idx, btn in enumerate(demo_btns):
        btn.click(
            load_demo,
            inputs=[gr.Number(value=idx, visible=False),
                    chatbot, scene_state, metrics_state],
            outputs=cmd_outputs + [add_status])

demo.launch(share=True, debug=True)

Gradio version: 3.50.2
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://97e1715ffe33d7f3d0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://97e1715ffe33d7f3d0.gradio.live


In [14]:
import os
os.chdir('/content/compose-axion-ax-hackathon-2026-full-solution-template')
!git add src/perception.py src/simulation/pybullet_env.py
!git commit -m "fix: remove Shape.BLOCK references in perception and pybullet_env"
!git push origin main

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@6216282554b8.(none)')
fatal: could not read Username for 'https://github.com': No such device or address
